In [1]:
import os

In [2]:
%pwd

'c:\\Users\\nalin\\CluckCare\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\nalin\\CluckCare'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from cluckcare.constants import *
from cluckcare.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
CONFIG_FILEPATH = Path("config/config.yaml"),
        config_filepath = CONFIG_FILEPATH,
        params_filepath = PARAMS_FILEPATH):

        self.config = read_yaml(CONFIG_FILEPATH)
        self.params = read_yaml(PARAMS_FILEPATH)

        create_directories([self.config.artifacts_root])
        self.params = read_yaml(PARAMS_FILEPATH)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config
      

In [8]:
import os
import urllib.request as request
import zipfile
from cluckcare import logger
from cluckcare.utils.common import get_size 

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-17 16:06:08,435]: yaml file: config\config.yaml loaded successfully
[2026-10-17 16:06:08,435]: yaml file: params.yaml loaded successfully
[2026-10-17 16:06:08,442]: created directory at: artifacts
[2026-10-17 16:06:08,442]: yaml file: params.yaml loaded successfully
[2026-10-17 16:06:08,442]: created directory at: artifacts
[2026-10-17 16:06:08,442]: created directory at: artifacts/data_ingestion
[2026-10-17 16:06:12,089]: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: C47C:1A2258:4B2464:5BB42C:6AD34F99
x-github-edge-region: centralindia
Accept-Ranges: bytes
Dat